# Financial Fraud Detection — OpenShift Inference

This notebook runs inference against a Triton server deployed on OpenShift via Helm.

**Prerequisites:**
* Preprocessing and training completed on the cluster (`deploy-preprocess.sh`, `deploy-training.sh`)
* Inference server deployed and running (`deploy-inference.sh`)
* The Triton route is accessible from this machine

## Setup

In [ ]:
!pip install -q 'tritonclient[all]' requests numpy scikit-learn matplotlib pandas

In [ ]:
import subprocess
import numpy as np
import tritonclient.http as httpclient
from tritonclient.http import InferInput, InferRequestedOutput

## Connect to Triton on OpenShift
The endpoint is auto-detected from the OpenShift route.

In [ ]:
TRITON_HOST = subprocess.check_output(
    ["oc", "get", "route", "fraud-triton", "-o", "jsonpath={.spec.host}"],
    text=True
).strip()
TRITON_URL = f"https://{TRITON_HOST}"
print(f"Triton endpoint: {TRITON_URL}")

In [ ]:
import requests

resp = requests.get(f"{TRITON_URL}/v2/health/ready")
print(f"Health check: {resp.status_code} {'OK' if resp.ok else 'FAILED'}")

resp = requests.get(f"{TRITON_URL}/v2/models/prediction_and_shapley/ready")
print(f"Model ready:  {resp.status_code} {'OK' if resp.ok else 'FAILED'}")

## Inference helper
Prepares inputs and sends an inference request to the Triton server over HTTPS.

In [ ]:
def prepare_and_send_inference_request(data):
    client = httpclient.InferenceServerClient(
        url=TRITON_HOST, ssl=True, ssl_context_factory=None
    )

    inputs = []
    def _add_input(name, arr, dtype):
        inp = InferInput(name, arr.shape, datatype=dtype)
        inp.set_data_from_numpy(arr)
        inputs.append(inp)

    for key, value in data.items():
        if key.startswith("x_"):
            dtype = "FP32"
        elif key.startswith("feature_mask_"):
            dtype = "INT32"
        elif key.startswith("edge_feature_mask_"):
            dtype = "INT32"
        elif key.startswith("edge_index_"):
            dtype = "INT64"
        elif key.startswith("edge_attr_"):
            dtype = "FP32"
        elif key == "COMPUTE_SHAP":
            dtype = "BOOL"
        else:
            continue

        _add_input(key, value, dtype)

    outputs = [InferRequestedOutput("PREDICTION")]
    for key in data:
        if key.startswith("x_"):
            node = key[len("x_"):]
            outputs.append(InferRequestedOutput(f"shap_values_{node}"))
        elif key.startswith("edge_attr_"):
            edge_name = key[len("edge_attr_"):]
            outputs.append(InferRequestedOutput(f"shap_values_{edge_name}"))

    response = client.infer(
        "prediction_and_shapley",
        inputs=inputs,
        request_id=str(1),
        outputs=outputs,
        timeout=3000
    )

    result = {"PREDICTION": response.as_numpy("PREDICTION")}
    for key in data:
        if key.startswith("x_"):
            node = key[len("x_"):]
            result[f"shap_values_{node}"] = response.as_numpy(f"shap_values_{node}")
        if key.startswith("edge_attr_"):
            edge_name = key[len("edge_attr_"):]
            result[f"shap_values_{edge_name}"] = response.as_numpy(f"shap_values_{edge_name}")

    return result

## Quick test with random data

In [ ]:
def make_example_request():
    num_merchants = 5
    num_users = 7
    num_edges = 3
    merchant_feature_dim = 24
    user_feature_dim = 13
    user_to_merchant_feature_dim = 38

    return {
        "x_merchant": np.random.randn(num_merchants, merchant_feature_dim).astype(np.float32),
        "x_user": np.random.randn(num_users, user_feature_dim).astype(np.float32),
        "COMPUTE_SHAP": np.array([False], dtype=np.bool_),
        "feature_mask_merchant": np.random.randint(0, 2, size=(merchant_feature_dim,), dtype=np.int32),
        "feature_mask_user": np.random.randint(0, 2, size=(user_feature_dim,), dtype=np.int32),
        "edge_index_user_to_merchant": np.vstack([
            np.random.randint(0, num_users, size=(num_edges,)),
            np.random.randint(0, num_merchants, size=(num_edges,))
        ]).astype(np.int64),
        "edge_attr_user_to_merchant": np.random.randn(num_edges, user_to_merchant_feature_dim).astype(np.float32),
        "edge_feature_mask_user_to_merchant": np.random.randint(0, 2, size=(user_to_merchant_feature_dim,), dtype=np.int32)
    }

In [ ]:
data = make_example_request()
result = prepare_and_send_inference_request(data)
print("Predictions:", result["PREDICTION"])

## Inference with real test data
This requires the preprocessed test data to be available locally under `../data/TabFormer/gnn/test_gnn`.
If you haven't run preprocessing locally, you can skip this section — the random data test above confirms the server works.

In [ ]:
import os
import sys

src_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.insert(0, src_dir)

data_root_dir = os.path.abspath('../data/TabFormer/')
gnn_data_dir = os.path.join(data_root_dir, "gnn")

In [ ]:
from preprocess_TabFormer_lp import preprocess_data, load_hetero_graph

test_data = load_hetero_graph(os.path.join(gnn_data_dir, "test_gnn"))

### Prediction without Shapley values

In [ ]:
compute_shap = False
result = prepare_and_send_inference_request(
    test_data | {"COMPUTE_SHAP": np.array([compute_shap], dtype=np.bool_)}
)
result['PREDICTION']

### Evaluate performance on test data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, ConfusionMatrixDisplay
)

def compute_score_for_batch(y, predictions, decision_threshold=0.5):
    y_pred = (predictions > decision_threshold).astype(int)

    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)

    classes = ['Non-Fraud', 'Fraud']
    columns = pd.MultiIndex.from_product([["Predicted"], classes])
    index = pd.MultiIndex.from_product([["Actual"], classes])

    conf_mat = confusion_matrix(y, y_pred)
    cm_df = pd.DataFrame(conf_mat, index=index, columns=columns)
    print(cm_df)

    disp = ConfusionMatrixDisplay.from_predictions(y, y_pred, display_labels=classes)
    disp.ax_.set_title('Confusion Matrix')
    plt.show()

    print("----Summary---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

In [ ]:
decision_threshold = 0.5
y = test_data['edge_label_user_to_merchant'].to_numpy(dtype=np.int32)
compute_score_for_batch(y, result['PREDICTION'], decision_threshold)

### Compute Shapley values
NOTE: Shapley computation is expensive — it only computes SHAP values for the first transaction.

In [ ]:
compute_shap = True
result_with_shap = prepare_and_send_inference_request(
    test_data | {"COMPUTE_SHAP": np.array([compute_shap], dtype=np.bool_)}
)

for key in result_with_shap:
    if key.startswith('shap_'):
        print(f'{key} : {result_with_shap[key]}')